# 11 · Ports, topology and margins: the network as data

Tutorial 10 wrote every connection by hand. That does not scale to "two lane motors per rotor, two cross-strapped buses, a battery split
into isolated strings, a step-up gearbox per engine, and failure cases that switch units off". The `core/` layer (ports, topology,
margins: 317 lines) and `powertrain/` helpers (`ports.py`, `topologies.py`, `compatibility.py`, `redundancy.py`) describe the network
**as data**: typed ports, named instances with counts, buses and connections. From that description the code derives connection
equations, per-component limits, masses at installed positions, and degraded states.

The topology is a *description only*: no variables, no constraints, no execution order, no solving (`core/topology.py` docstring). It is
lighter than an OpenMDAO `Group` by design.

**In this notebook**
1. Ports: domains, directions, port values; which ports each component has.
2. `Topology`: instances with counts, buses, connections, and the wiring errors it rejects at build time.
3. `connection_residuals`: the equations a topology implies, and tutorial 10's point rebuilt from them.
4. Multiplicity: `count = n`, combiners and splitters, buses; four rotor strings with the same variables as one.
5. `operating_margins` and `design_margins`: every limit, named.
6. `build_series_hybrid` and the `RedundancyLayer`.
7. Degraded states: failures are flight-condition inputs, not new topologies.
8. Residuals versus "by construction": what the production flight point does.

**Run time:** a few seconds.

In [1]:
from tutorial_helpers import *
import inspect
from dataclasses import replace
from aircraft_closure.core.ports import Domain, Direction, PortSpec, MechanicalPortValue, ElectricalPortValue
from aircraft_closure.core.topology import Topology, connection_residuals
from aircraft_closure.core.margins import margin_report
from aircraft_closure.powertrain.ports import port_specs_for
from aircraft_closure.powertrain.components.motor import Motor
from aircraft_closure.powertrain.components.generator import Generator
from aircraft_closure.powertrain.components.battery import Battery
from aircraft_closure.powertrain.components.turboshaft import SimpleTurboshaft
from aircraft_closure.powertrain.components.gearbox import Gearbox
from aircraft_closure.powertrain.components.propulsor import ActuatorDiskPropulsor
from aircraft_closure.powertrain.components.protection import ProtectionUnit
from aircraft_closure.powertrain.components.cable import Cable

## 1. Ports

A port has a **domain** (mechanical: speed and torque; electrical: voltage and current; fuel: fuel flow) and a nominal **direction** of
power or fuel flow relative to its component (IN or OUT). Port *values* are small dataclasses whose fields may be numbers or
expressions. Flow is positive in the nominal direction: a motor's electrical current is positive when drawn, a generator's when
delivered, a battery's on discharge.

Port declarations live in `powertrain/ports.py`, **outside** the component classes, so the component contract stays
`get_mass / get_limits / evaluate` (tutorial 04). `port_specs_for` walks the class's MRO, so a subclass inherits its parent's ports:

In [2]:
for component in (Motor(), Generator(), Battery(), SimpleTurboshaft(), Gearbox(), ActuatorDiskPropulsor(), ProtectionUnit(), Cable()):
    ports = ", ".join(f"{p.name} ({p.domain.value}, {p.direction.value})" for p in port_specs_for(component))
    print(f"{type(component).__name__:<22} {ports}")

Motor                  electrical (electrical, in), shaft (mechanical, out)
Generator              shaft (mechanical, in), electrical (electrical, out)
Battery                electrical (electrical, out)
SimpleTurboshaft       fuel (fuel, in), shaft (mechanical, out)
Gearbox                shaft_in (mechanical, in), shaft_out (mechanical, out)
ActuatorDiskPropulsor  shaft (mechanical, in)
ProtectionUnit         input (electrical, in), output (electrical, out)
Cable                  input (electrical, in), output (electrical, out)


## 2. `Topology`

`add(name, component, ports, count)` registers an instance; `add_bus(name, count)` an electrical junction; `connect(a, b, combine=False)`
joins two ports (one OUT to one IN, same domain) or a port and a bus. **Every connection is validated immediately**, long before any
solve:

In [3]:
c = dict(turboshaft=SimpleTurboshaft(), generator=Generator(), motor=Motor(), gearbox=Gearbox(), propulsor=ActuatorDiskPropulsor())

def try_wiring(label, build):
    t = Topology()
    for name, component in c.items():
        t.add(name, component, port_specs_for(component), count=4 if name == "motor" else 2)
    t.add_bus("bus")
    try:
        build(t)
        print(f"accepted  {label}")
        return True
    except ValueError as error:
        print(f"REJECTED  {label}\n          -> {error}")
        return False

ok = [try_wiring("shaft port to an electrical bus", lambda t: t.connect("turboshaft.shaft", "bus")),
      try_wiring("two OUT ports joined", lambda t: t.connect("turboshaft.shaft", "motor.shaft")),
      try_wiring("4 motors on 2 gearboxes, no combine", lambda t: t.connect("motor.shaft", "gearbox.shaft_in")),
      try_wiring("4 motors on 2 gearboxes as a combiner", lambda t: t.connect("motor.shaft", "gearbox.shaft_in", combine=True)),
      try_wiring("the same port connected twice",
                 lambda t: (t.connect("gearbox.shaft_out", "propulsor.shaft"), t.connect("gearbox.shaft_out", "propulsor.shaft")))]
check("only the combiner is accepted", ok == [False, False, False, True, False])

REJECTED  shaft port to an electrical bus
          -> Cannot connect mechanical port 'turboshaft.shaft' to electrical bus 'bus'.
REJECTED  two OUT ports joined
          -> Direct connections join one OUT and one IN port: 'turboshaft.shaft' and 'motor.shaft' are both out.
REJECTED  4 motors on 2 gearboxes, no combine
          -> Direct connection between counts 4 and 2 needs combine=True (a combiner or splitter) or a bus.
accepted  4 motors on 2 gearboxes as a combiner
REJECTED  the same port connected twice
          -> Port 'gearbox.shaft_out' is already connected; use a bus for junctions.
PASS  only the combiner is accepted


## 3. The equations a topology implies

`connection_residuals(topology, port_values)` returns labelled residuals for the caller to drive to zero:

- **direct connection**: every field of source and target equal (speed and torque, or voltage and current);
- **combiner/splitter** (`combine=True`): efforts (speed, voltage) equal; flow conserved,
  $n_\text{source}\,\text{flow}_\text{source} = n_\text{target}\,\text{flow}_\text{target}$;
- **bus**: all voltages equal to the first port's; Kirchhoff's current law weighted by counts, OUT ports delivering and IN ports drawing.

The caller supplies a value for every connected port. `examples/series_hybrid_point.py` rebuilds tutorial 10's point this way: it makes
the bus voltage, the turboshaft torque and the rotor speed and torque variables (the hand version wrote them by substitution), applies
every residual as an equality, and replaces the hand-written limit list by `operating_margins`. Let's look at its residuals and compare
the answers:

In [4]:
from examples.series_hybrid_point import build_point_problem, solve_topology_point, build_reference_topology
from examples.series_hybrid_point_explicit import solve_reference_point

opti = asb.Opti()
problem = build_point_problem(opti, count_rotors=1, hybridization_electric=0.2)
opti.subject_to(problem.rotor.thrust_N == 5000.0)
solution = opti.solve(verbose=False)
for residual in problem.residuals:
    print(f"{residual.label:<52} = {solution.value(residual.value):+.2e}")
print(describe_opti(opti))
explicit = solve_reference_point()
check("the topology route reproduces the hand-written point",
      abs(solution.value(problem.engine.fuel_flow_kg_s) - explicit.fuel_flow_kg_s) < 1e-9)

turboshaft.shaft->generator.shaft speed_rad_s        = +0.00e+00
turboshaft.shaft->generator.shaft torque_Nm          = +8.53e-14
motor.shaft->gearbox.shaft_in speed_rad_s            = +0.00e+00
motor.shaft->gearbox.shaft_in torque_Nm              = +0.00e+00
gearbox.shaft_out->propulsor.shaft speed_rad_s       = +0.00e+00
gearbox.shaft_out->propulsor.shaft torque_Nm         = +2.27e-13
bus battery.electrical voltage_V                     = +0.00e+00
bus motor.electrical voltage_V                       = +0.00e+00
bus current_A                                        = -2.10e-10
{'variables': 8, 'constraints': 29, 'equalities': 8, 'inequalities': 21}


PASS  the topology route reproduces the hand-written point


Same answer, more variables (8 instead of 4) and more equalities (8 instead of 4): the price of generality. Section 8 explains why the production flight
point writes connections by construction instead, and uses the topology for everything else.

## 4. Multiplicity

An instance with `count = n` is **n identical copies sharing one set of port values**: symmetric operation. Mass, bus current, heat
loads and margins scale by n; there is only one set of variables. `build_reference_topology(count_rotors)` scales the shared source
side (generator, battery, turboshaft) with the number of rotor strings. Four strings then solve with the same per-string state as one:

In [5]:
one, four = solve_topology_point(count_rotors=1), solve_topology_point(count_rotors=4)
print(f"1 string : motor current {one.current_motor_A:.3f} A, bus {one.voltage_bus_V:.3f} V, total motor current {one.current_motors_total_A:.2f} A")
print(f"4 strings: motor current {four.current_motor_A:.3f} A, bus {four.voltage_bus_V:.3f} V, total motor current {four.current_motors_total_A:.2f} A")
check("four identical strings: same per-motor state, four times the bus current",
      abs(one.current_motor_A - four.current_motor_A) < 1e-6 and abs(four.current_motors_total_A - 4 * one.current_motor_A) < 1e-6)

1 string : motor current 120.081 A, bus 798.799 V, total motor current 120.08 A
4 strings: motor current 120.081 A, bus 798.799 V, total motor current 480.32 A
PASS  four identical strings: same per-motor state, four times the bus current


The limitation is the flip side: **asymmetric** states (one rotor degraded, the other healthy, with the roll trim that implies) cannot be
represented. The Halo applies failures to both rotors at once, which is conservative for power and heat (README, next steps).

A worked combiner: two lane motors at 1,000 N·m each feed one gearbox input of 2,000 N·m.

In [6]:
t = Topology()
t.add("motor", Motor(), port_specs_for(Motor()), count=4)
t.add("gearbox", Gearbox(), port_specs_for(Gearbox()), count=2)
t.connect("motor.shaft", "gearbox.shaft_in", combine=True)
values = {"motor.shaft": MechanicalPortValue(150.0, 1000.0), "gearbox.shaft_in": MechanicalPortValue(150.0, 2000.0)}
for residual in connection_residuals(t, values):
    print(f"{residual.label:<45} = {residual.value}")

motor.shaft->gearbox.shaft_in speed_rad_s     = 0.0
motor.shaft->gearbox.shaft_in torque_Nm (total) = 0.0


## 5. Margins from the topology

`operating_margins(topology, port_values, atmosphere, components=None)` (`powertrain/compatibility.py`) loops over the instances and
returns every per-component limit as a named margin:

| Component | Operating margins |
|---|---|
| Motor, generator | torque ≤ max; speed ≤ max; bus voltage in the machine's window; shaft power ≤ rating (*unless* a thermal model is on: then temperature) |
| `Battery` | discharge and charge power ≤ ratings |
| `EquivalentCircuitBattery` | discharge and charge current ≤ ratings; terminal voltage in $N_s \times [2.5, 4.2]$ V |
| Turboshaft | shaft power ≤ power available *at this point's atmosphere* |
| Gearbox, rotor | input or shaft power ≤ rating |
| Cable, protection | current² ≤ rating²; cable partial discharge at the point's pressure |
| Inverter, DC/DC | power ≤ rating; DC voltage window |

The point above constrained them all ≥ 0; the report names what binds:

In [7]:
for entry in margin_report(problem.margins, solution.value)[:6]:
    print(f"{entry.value:+.4f}  {entry.label}")

+0.0795  motor power_shaft_W
+0.0795  gearbox power_input_W
+0.1071  propulsor power_shaft_W
+0.1124  generator max_voltage_V
+0.1124  motor max_voltage_V
+0.1993  generator power_shaft_W


`design_margins(topology)` is the other kind: **adjacent-rating** checks from component parameters only. Does the gearbox tolerate the
motor's maximum torque × count? Is every machine's voltage window inside the range the battery can hold the bus to? Is the loss-free
source power on each bus at least the demand? The Halo sizing does not constrain these (it relies on operating margins at the points it
flies), but they are a useful audit of a sized design:

In [8]:
from aircraft_closure.powertrain.compatibility import design_margins
topology_reference = build_reference_topology(count_rotors=1)
for m in sorted(design_margins(topology_reference), key=lambda m: float(m.value))[:8]:
    print(f"{float(m.value):+8.3f}  {m.label}")

  -0.500  turboshaft.shaft->generator.shaft max_power_W
  +0.000  motor.shaft->gearbox.shaft_in max_power_W
  +0.030  gearbox.shaft_out->propulsor.shaft max_power_W
  +0.108  bus battery.electrical->generator.electrical max_voltage_V
  +0.108  bus battery.electrical->motor.electrical max_voltage_V
  +0.984  bus battery.electrical->generator.electrical min_voltage_V
  +0.984  bus battery.electrical->motor.electrical min_voltage_V
  +1.000  bus power_W (loss-free)


Negative design margins on default components are expected: the defaults are not a consistent design. A real design would be audited
after sizing.

## 6. `build_series_hybrid` and the `RedundancyLayer`

`powertrain/topologies.py: build_series_hybrid(motor, generator, battery, turboshaft, gearbox, propulsor, count_rotors,
count_turbogenerators, generator_gearbox=None, electrical=None, redundancy=None)` builds the reference network:

```
m × (turboshaft [→ generator_gearbox] → generator) → bus ← battery [→ string contactors]
bus → (n × lanes) × motor ⇒ n × gearbox → n × rotor          (⇒ a combiner when lanes > 1)
```

`ElectricalLayer` (optional) inserts inverters, cables and contactors in every feeder. `RedundancyLayer` adds lane motors per rotor (a
combiner on the gearbox input), `count_buses` cross-strapped buses (the bus count) with normally open ties, and isolated battery
strings (a splitter from the pack through `protection_string` contactors). Hardware is only added for counts above one. The Halo's
architecture:

In [9]:
from aircraft_closure.powertrain.topologies import build_series_hybrid, RedundancyLayer
tie, cable_tie = ProtectionUnit(max_current_A=600.0), Cable(length_m=2.0, max_current_A=600.0)
redundancy = RedundancyLayer(count_lanes=2, count_buses=2, count_strings_battery=2, protection_string=ProtectionUnit(max_current_A=700.0),
                             protection_bus_tie=tie, cable_bus_tie=cable_tie)
halo_like = build_series_hybrid(Motor(), Generator(), Battery(), SimpleTurboshaft(), Gearbox(), ActuatorDiskPropulsor(),
                                count_rotors=2, count_turbogenerators=2, generator_gearbox=Gearbox(reduction_ratio=0.1),
                                redundancy=redundancy)
print("instances:")
for name, instance in halo_like.instances.items():
    print(f"   {name:<20} {type(instance.component).__name__:<22} x{instance.count}")
print("buses:", {name: bus.count for name, bus in halo_like.buses.items()})
print("connections:")
for connection in halo_like.connections:
    print(f"   {connection.source:<28} -> {connection.target:<22} {'(combine)' if connection.combine else ''}")
check("2 lanes per rotor: 4 lane motors on 2 rotors", halo_like.instances["motor"].count == 4)

instances:
   turboshaft           SimpleTurboshaft       x2
   generator            Generator              x2
   battery              Battery                x1
   motor                Motor                  x4
   gearbox              Gearbox                x2
   propulsor            ActuatorDiskPropulsor  x2
   generator_gearbox    Gearbox                x2
   protection_string    ProtectionUnit         x2
   protection_bus_tie   ProtectionUnit         x1
   cable_bus_tie        Cable                  x1
buses: {'bus': 2}
connections:
   turboshaft.shaft             -> generator_gearbox.shaft_in 
   generator_gearbox.shaft_out  -> generator.shaft        
   battery.electrical           -> protection_string.input (combine)
   protection_bus_tie.output    -> cable_bus_tie.input    
   generator.electrical         -> bus                    
   protection_string.output     -> bus                    
   motor.electrical             -> bus                    
   motor.shaft                 

Note the bus tie: `protection_bus_tie.output → cable_bus_tie.input` and nothing else. Its outer ports are **boundaries**: in normal operation
the tie is open and carries no current. The turboshaft's fuel port is also a boundary (fuel flow is read from the engine and integrated by the
mission).

## 7. Degraded states are flight-condition inputs

A failure is not a different network. It is the **same** network with fewer active units at one flight point. `redundancy.py` reads the counts
from the topology and validates a condition's failure state:

- `active_lane_count` (per rotor): fewer lanes carry the gearbox torque;
- `count_buses_failed`: that bus's lanes are lost and its sources feed the other bus through a tie;
- `active_battery_string_count`: the pack runs on fewer strings at the same SOC (`battery_with_strings` scales capacity, ratings and
  conductance).

In [10]:
from aircraft_closure.powertrain.redundancy import redundancy_counts, degraded_state, battery_with_strings
from aircraft_closure.performance.flight_point import FlightCondition
print(redundancy_counts(halo_like))
for label, kwargs in (("all units", {}), ("lane out", dict(active_lane_count=1)), ("bus out", dict(count_buses_failed=1)),
                      ("string out", dict(active_battery_string_count=1))):
    state = degraded_state(halo_like, FlightCondition(mode="hover", **kwargs))
    print(f"{label:<10} lanes active per rotor {state.count_lanes_active}, strings {state.count_strings_active}, "
          f"buses failed {state.count_buses_failed}, string fraction {state.fraction_strings_active}")
try:
    degraded_state(halo_like, FlightCondition(mode="hover", active_lane_count=1, count_buses_failed=1))
except ValueError as error:
    print("lane out + bus out ->", error)
pack = Battery()
half = battery_with_strings(pack, 0.5)
print(f"string out: capacity {pack.energy_capacity_J / 3.6e6:.1f} -> {half.energy_capacity_J / 3.6e6:.1f} kWh, "
      f"resistance {pack.resistance_ohm:.3f} -> {half.resistance_ohm:.3f} ohm")

RedundancyCounts(count_lanes=2, count_buses=2, count_strings_battery=2, count_ties=1)
all units  lanes active per rotor 2, strings 2, buses failed 0, string fraction 1.0
lane out   lanes active per rotor 1, strings 2, buses failed 0, string fraction 1.0
bus out    lanes active per rotor 1, strings 2, buses failed 1, string fraction 1.0
string out lanes active per rotor 2, strings 1, buses failed 0, string fraction 0.5
lane out + bus out -> The failure state leaves no lane on a rotor.
string out: capacity 10.0 -> 5.0 kWh, resistance 0.050 -> 0.100 ohm


So "survive a bus failure" is one more `FlightCondition(count_buses_failed=1)` point in the same sizing problem, with the same components and
variables shared with every other point. That is how the Halo's failure hovers size its motors (bus out) and battery (engine out).

## 8. Residuals versus "by construction"

The production flight point (`build_flight_point`, tutorial 15) **does not apply `connection_residuals`**. It writes the connections by
construction: one bus-voltage expression for every port on the bus, motor speed = gear ratio × rotor speed, gearbox torque split over the active
lanes, and the bus balance as two power equalities. It still builds the full `port_values` dictionary, but only to evaluate
`operating_margins`. Fewer variables and equalities, better conditioning.

There is a second, deeper reason. `connection_residuals` uses the **installed** counts. In a lane-out point, one active lane per rotor carries
the whole gearbox torque $Q$, so the combiner residual would read $4Q - 2Q \ne 0$ although the physics is right. Failure states live in the flight
condition, and the flight point applies them; the topology describes the installed network.

What the topology is used for in the sizing:

1. validation at build time (domains, directions, counts);
2. multiplicity: installed counts, and the base for active counts in failure states;
3. lookup of components by instance name (`topology.instances["motor"].component`);
4. mass and CG: `PowertrainInstallation` books count × `get_mass()` at each installed location (tutorial 12);
5. operating margins at every point;
6. names of heat loads and margins.

## Summary

- Ports are typed (domain, direction); their values are numbers or expressions; declarations sit outside the components.
- A `Topology` holds instances with counts, buses (with counts) and validated connections; it never owns variables or solving.
- `connection_residuals` gives the implied equations; `operating_margins` and `design_margins` give every limit, named.
- `count = n` means n identical copies with one set of variables: cheap, exact for symmetric operation, unable to represent asymmetry.
- Failures are flight-condition inputs applied by the flight point; the production point writes connections by construction.

## Check yourself

<details><summary>1. What does a bus with <code>count=2</code> mean?</summary>

Two identical cross-strapped buses, each with an equal share of the attached copies, lumped as one junction under symmetric operation. The count matters to callers modelling a bus failure (one bus's lanes lost, its sources through a tie).
</details>

<details><summary>2. You add a new component type (say, a supercapacitor) to the network. What must you add besides the class?</summary>

Its port declaration in `powertrain/ports.py`; its operating margins in `compatibility.py` (otherwise `operating_margins` raises for an unknown type); its physics in `build_flight_point` (the topology alone will not couple it); and its location in the `PowertrainInstallation` (which requires every instance to be located).
</details>

In [11]:
check_summary()

4 of 4 checks passed
